WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

plt.rcParams.update(_viz.default_plot_settings)
jax.config.update("jax_enable_x64", True)

### Geometry

In [ ]:
rod_length = 10.0  # [m]

fiber_offset = 0.1  # fiber dist from center [m]
fiber_radius = 0.02 # radius of each fiber [m]

In [ ]:
# Four fibers in a plus: (+c,0), (-c,0), (0,+c), (0,-c) in the (d1, d2) plane.
fiber_offsets = np.array(
  [
    [fiber_offset, 0.0],
    [-fiber_offset, 0.0],
    [0.0, fiber_offset],
    [0.0, -fiber_offset],
  ]
)

num_fibers = fiber_offsets.shape[0]
area_per_fiber = np.pi * fiber_radius**2

num_elems = 40
num_nodes = num_elems + 1

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda k: jnp.array([k * rod_length, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, 1.0, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(fiber_offsets, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),  # treat fibers as points
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

In [ ]:
# %matplotlib qt5

nodal_offsets = rod.fibers.nodal_offsets(rod.elem_arc_lengths)
fiber_xyz = _mesher.compute_fiber_positions(
  node_coords_ref=rod.node_coords,
  directors_ref=rod.directors_0,
  fiber_offsets=nodal_offsets,
)

ax = _viz.plot_rod_centerline(rod.node_coords)
ax = _viz.plot_rod_fibers(np.asarray(fiber_xyz), ax=ax)
ax.set_title("Undeformed rod and fiber bundle")
plt.show()

In [ ]:
moi_11 = np.sum(area_per_fiber * fiber_offsets[:, 0] ** 2)
moi_22 = np.sum(area_per_fiber * fiber_offsets[:, 1] ** 2)
area_total = num_fibers * area_per_fiber

In [ ]:
shear_modulus = 1.0e6  # G [Pa]
youngs_modulus = 3.0 * shear_modulus

bending_stiffness = youngs_modulus * moi_11  # EI
shear_stiffness = shear_modulus * area_total  # kappa*GA with kappa = 1

In [ ]:
print(f"rod.elem_areas[0] = {rod.elem_areas[0]:.6e}  vs  {area_total:.6e}")
print(f"rod.elem_moi_tensor[0,0,0] = {rod.elem_moi_tensor[0,0,0]:.6e} vs {moi_11:.6e}")
print(f"rod.elem_moi_tensor[0,1,1] = {rod.elem_moi_tensor[0,1,1]:.6e} vs {moi_22:.6e}")
print(f"reference axial stretch nu = {rod.ref_kinematics.strain[0, 2]:.12f}  (want 1)")
print(f"reference curvature = {rod.ref_kinematics.curvature[0]}  (want 0)")

### BC

In [ ]:
tip_load = 8.0 # [N]
num_steps = 30

clamped_node = np.array([0])
tip_node = np.array([num_nodes - 1])

clamped_dofs = (
  _utils.Translation.X, _utils.Translation.Y, _utils.Translation.Z,
  _utils.Rotation.THETA_X, _utils.Rotation.THETA_Y, _utils.Rotation.THETA_Z,
)

# Every BC must carry a schedule of length num_steps. The clamps are zero at every
# step, so their schedule is just padding to satisfy that contract.
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[clamped_node for _ in range(num_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

t = np.arange(1, num_steps + 1) / num_steps
loads = tip_load * t**2  # quadratic ramp to avoid shock loading
load = _bc.NeumannBC(
  values=(_utils.Translation.Z, [np.array([p]) for p in loads]),
  node_targets=[tip_node for _ in range(num_steps)],
  name="tip_load",
)

boundary_conditions = _bc.process_boundary_conditions(rod, clamp + [load])
print(f"num_steps = {boundary_conditions.num_steps}")

In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 300,
    "threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 15,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
    "abs_threshold": 1e-6,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-8},
)


problem = _fe.CosseratProblem(rod=rod, bc=boundary_conditions, nr_solver=nr_solver)

In [ ]:
shear_moduli = jnp.full((num_elems, num_fibers), shear_modulus)
u_star, _ = problem.solve(u0=jnp.zeros(rod.num_dofs), shear_moduli=shear_moduli)

u_nodal = np.asarray(u_star).reshape(num_nodes, rod.num_dofs_per_node)
deflection = u_nodal[:, _utils.Translation.Z.value]
print(f"tip defl. = {deflection[-1]:.6e} m   ({deflection[-1]/rod_length:.3%} of L)")

In [ ]:
def timoshenko_deflection(x, load, length, bending_stiffness, shear_stiffness):
  """Timoshenko cantilever deflection under a transverse tip load."""
  bending = load * x**2 * (3.0 * length - x) / (6.0 * bending_stiffness)
  shear = load * x / shear_stiffness
  return bending, shear


x_nodes = np.asarray(rod.node_coords[:, 0])
bend, shear = timoshenko_deflection(
  x_nodes, tip_load, rod_length, bending_stiffness, shear_stiffness
)
analytical = bend + shear

print(f"analytical tip = {analytical[-1]:.6e} m")
print(f"bending part = {bend[-1]:.6e}  ({bend[-1]/analytical[-1]:.2%})")
print(f"shear part   = {shear[-1]:.6e}  ({shear[-1]/analytical[-1]:.2%})")
print(f"crops tip      = {deflection[-1]:.6e} m")
print(f"relative error = {abs(deflection[-1]-analytical[-1])/analytical[-1]:.4%}")

In [ ]:
fig, (ax_profile, ax_error) = plt.subplots(1, 2, figsize=(11, 4))

ax_profile.plot(x_nodes, analytical, "k-", lw=2, label="Timoshenko")
ax_profile.plot(x_nodes, bend, "k--", lw=1, alpha=0.5, label="Euler-Bernoulli only")
ax_profile.plot(x_nodes, deflection, "o", ms=4, mfc="none", label="CROPS")
ax_profile.set_xlabel("$x$ [m]")
ax_profile.set_ylabel("$w(x)$ [m]")
ax_profile.legend()

ax_error.plot(x_nodes[1:], np.abs(deflection[1:] - analytical[1:]) / analytical[1:])
ax_error.set_xlabel("$x$ [m]")
ax_error.set_ylabel("relative error")
ax_error.set_yscale("log")

fig.tight_layout()
plt.show()